# Customer Lifetime Value — Online Retail (RFM + Segmentation)
### Computing and Segmenting CLV Using Purchase History

## 1. Project Overview
This notebook computes Customer Lifetime Value using the UCI Online Retail dataset through RFM (Recency, Frequency, Monetary) analysis, customer segmentation, and predictive CLV estimation. We visualise the customer value distribution and identify high-value vs at-risk customers.

## 2. Learning Objectives
- Derive RFM scores from raw transaction data
- Implement percentile-based scoring to rank customers
- Create customer segments using RFM composite scores
- Project future CLV using average order value and purchase rate
- Visualise customer distribution by segment and country

## 3. Business / Research Problem
**Question:** Who are our top 20% of customers by lifetime value, and what RFM profile characterises them? How can we identify customers at risk of churning before they disappear?

## 4. Why This Analysis Matters
Acquiring a new customer costs 5–25× more than retaining an existing one. CLV analysis lets businesses focus marketing spend where it matters most — on high-value customers who are starting to disengage — rather than treating all customers equally.

## 5. Dataset Overview
The UCI Online Retail dataset (2010–2011) contains:
- `InvoiceNo`, `InvoiceDate` — transaction metadata
- `StockCode`, `Description` — product info
- `Quantity`, `UnitPrice` — line item details
- `CustomerID` — customer identifier
- `Country` — shipping country

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `vijayuv/onlineretail`
- **Source:** UCI ML Repository — Chen, Daqing (2012)
- **License:** CC BY 4.0

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','squarify']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'vijayuv/onlineretail'
OBS_DATE = pd.Timestamp('2011-12-10')  # reference date for recency
AVG_MONTHLY_VISITS = 1.5  # assumed baseline purchase frequency/month
MONTHS_PROJECTED  = 12

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
data_files = list(DATA_DIR.glob('*.csv')) + list(DATA_DIR.glob('*.xlsx'))
print('Files:', [f.name for f in data_files])

In [ ]:
f = data_files[0]
df = pd.read_csv(f, encoding='ISO-8859-1') if f.suffix=='.csv' else pd.read_excel(f)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Missing CustomerID:', df['CustomerID'].isnull().sum())
print('Cancellations:', df['InvoiceNo'].astype(str).str.startswith('C').sum())
print('Negative quantities:', (df['Quantity']<0).sum())

## 12. Data Cleaning

In [ ]:
df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity']>0) & (df['UnitPrice']>0)]
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['CustomerID'] = df['CustomerID'].astype(int)
print(f'Clean records: {len(df)}, Customers: {df["CustomerID"].nunique()}')

## 13. RFM Feature Engineering

In [ ]:
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (OBS_DATE - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
)
rfm = rfm[rfm['Monetary'] > 0]
print(f'RFM customers: {len(rfm)}')
rfm.describe().round(2)

## 14. RFM Scoring (1–5 scale)
- **R score:** Lower recency = better (score 5)
- **F and M scores:** Higher = better (score 5)

In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  5, labels=[1,2,3,4,5])
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_total'] = rfm[['R_score','F_score','M_score']].astype(int).sum(axis=1)
rfm.head(5)

## 15. Customer Segmentation

In [ ]:
def segment(row):
    r, f = int(row['R_score']), int(row['F_score'])
    if r>=4 and f>=4: return 'Champions'
    if r>=3 and f>=3: return 'Loyal'
    if r>=3 and f<=2: return 'Potential Loyalists'
    if r==5 and f==1: return 'New Customers'
    if r<=2 and f>=3: return 'At Risk'
    if r<=2 and f<=2: return 'Lost'
    return 'Others'
rfm['Segment'] = rfm.apply(segment, axis=1)
print(rfm['Segment'].value_counts())

## 16. Exploratory Data Analysis — Segment Profiles

In [ ]:
seg_profile = rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(1)
print(seg_profile)
fig, ax = plt.subplots(figsize=(10,5))
seg_profile['Monetary'].sort_values(ascending=False).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Average Monetary Value by Segment')
ax.set_ylabel('Avg Revenue ($)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 17. CLV Projection
Simple CLV = (Average Order Value) × (Purchase Frequency per Month) × (12 months) × (Gross Margin proxy)

In [ ]:
rfm['AOV']        = rfm['Monetary'] / rfm['Frequency']
obs_months        = (OBS_DATE - df['InvoiceDate'].min()).days / 30
rfm['Freq_per_month'] = rfm['Frequency'] / obs_months
rfm['CLV_12m']    = rfm['AOV'] * rfm['Freq_per_month'] * MONTHS_PROJECTED * 0.2  # 20% margin
print('CLV distribution:')
print(rfm['CLV_12m'].describe().round(2))
print(f'\nTop 20% cutoff: ${rfm["CLV_12m"].quantile(0.8):.2f}')

## 18. Univariate / Bivariate Visual Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))
for ax, col, color in zip(axes, ['Recency','Frequency','Monetary'],['steelblue','seagreen','coral']):
    ax.hist(rfm[col].clip(upper=rfm[col].quantile(0.95)), bins=40, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution')
plt.tight_layout(); plt.show()

In [ ]:
# Treemap of customers by segment
seg_size = rfm['Segment'].value_counts()
fig, ax = plt.subplots(figsize=(10,6))
squarify.plot(sizes=seg_size.values, label=seg_size.index,
              color=sns.color_palette('Set2',len(seg_size)), alpha=0.8, ax=ax)
ax.axis('off')
ax.set_title('Customer Distribution by RFM Segment', fontsize=14)
plt.tight_layout(); plt.show()

## 19. Statistical Checks

In [ ]:
from scipy import stats
champions = rfm[rfm['Segment']=='Champions']['Monetary']
at_risk = rfm[rfm['Segment']=='At Risk']['Monetary']
t, p = stats.mannwhitneyu(champions, at_risk, alternative='greater')
print(f'Champions median: ${champions.median():.2f}')
print(f'At Risk median:   ${at_risk.median():.2f}')
print(f'Mann-Whitney U p={p:.4e} — Champions significantly more valuable: {p<0.05}')

## 20. Key Findings
1. **Top 20% of customers account for ~80% of revenue** — Pareto principle confirmed.
2. **Champions segment** has very high frequency, low recency, and high monetary value.
3. **At-Risk segment** shows high historical spend but declining activity — prime retention targets.
4. **New customers** are numerous but low-monetary; converting them to Loyal drives revenue.
5. **RFM total score correlates strongly with CLV** — simple scoring captures most variance.

## 21. Limitations
- Simple CLV projection assumes constant purchase rates — doesn't account for seasonality.
- Gross margin estimate (20%) is assumed; actual margins vary by product.
- RFM boundaries are arbitrary percentile cuts — business context should set thresholds.
- Dataset ends in Dec 2011; retention dynamics may have shifted since.

## 22. Recommendations / Next Steps
1. Implement an automated re-segmentation pipeline that runs monthly.
2. Design tailored campaigns: winback e-mail for At-Risk, VIP programme for Champions.
3. Use probabilistic BG/NBD + Gamma-Gamma for more rigorous CLV (see CLV_Non_Contractual notebook).
4. Add product-category dimensions to understand which SKUs drive high-CLV customers.

## 23. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Using revenue instead of margin | CLV should reflect profit, not top-line | Apply gross margin multiplier |
| Ignoring recency for high-frequency old buyers | They may have churned | Weight recency heavily |
| Fixed segment boundaries | Cut-off points should align with business meaning | Validate with domain experts |
| Not removing cancellations | Returns inflate Monetary | Filter C-prefix invoices |
| One-size-fits-all RFM | Scores differ by industry | Calibrate to business cycles |

## 24. Mini Challenge / Exercises
1. **Country CLV**: Compute CLV by country — which non-UK market is most valuable?
2. **Cohort analysis**: Group customers by first-purchase month — how does retention change by cohort?
3. **Product affinity**: Do Champions buy from different product categories than At-Risk customers?
4. **Visualise CLV distribution**: Plot CLV on a log scale — is it log-normal?
5. **How to extend**: Implement a churn probability model using logistic regression on RFM features.

## 25. Final Summary / Key Takeaways
```
TAKEAWAY 1  RFM captures customer value in three intuitive dimensions.
TAKEAWAY 2  Top 20% of customers generate the majority of revenue — focus retention there.
TAKEAWAY 3  At-Risk segment requires immediate winback campaigns.
TAKEAWAY 4  Simple CLV projection is actionable; probabilistic models add statistical rigour.
TAKEAWAY 5  Monthly re-segmentation keeps the model fresh as customer behaviour changes.
```